# Hyper 2 - CNN + Mamba Experiment


In [1]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
import tifffile as tiff
from sklearn.metrics import classification_report
import keras_tuner as kt

In [2]:
CSV_PATH = r"C:\Users\rosha\Downloads\ailearn\HyperLeaf2024\train.csv"
DATA_DIR = r"C:\Users\rosha\Downloads\ailearn\HyperLeaf2024\images"
NUM_BANDS = 204
NUM_CLASSES = 4


## PreProcessing ##

In [3]:

df = pd.read_csv(CSV_PATH)

# Convert text cultivars ('Heerup', 'Kvium', etc.) to integers (0, 1, 2, 3)
image_ids = df['ImageId'].values

# The dataset uses one-hot encoded columns for the 4 classes
cultivar_cols = ['Heerup', 'Kvium', 'Rembrandt', 'Sheriff']

# Extract those 4 columns as a NumPy matrix and use argmax to convert the 1s into class indices (0, 1, 2, or 3)
labels = np.argmax(df[cultivar_cols].values, axis=1)

# Initialize empty arrays to hold our compressed 1D signatures
NUM_STATS=5
X_data = np.zeros((len(image_ids), NUM_BANDS,NUM_STATS), dtype=np.float32)

y_data = labels.astype(np.int32)

print("Processing hyperspectral cubes into 1D spectral vectors...")

for i, img_id in enumerate(image_ids):
    # 1. Update the extension to .tiff or .tif based on your folder
    file_path = os.path.join(DATA_DIR, f"{img_id:05d}.tiff")

    # 2. Load the multi-band TIFF
    cube = tiff.imread(file_path)

    # 3. CRITICAL: Fix the shape if bands are the first dimension
    # If shape is (204, Height, Width), change it to (Height, Width, 204)
    if cube.shape[0] == NUM_BANDS:
        cube = np.transpose(cube, (1, 2, 0))

    # Step A: Background Masking
    # Calculate the mean int
    # ensity per pixel across all 204 bands
    mean_intensity = np.mean(cube, axis=-1)

    # Define a threshold to isolate the leaf from the dark background
    threshold = np.percentile(mean_intensity, 15)
    leaf_mask = mean_intensity > threshold

    # Step B: Isolate leaf pixels and flatten spatially
    leaf_pixels = cube[leaf_mask]

    # Step C: Spatial Pooling (Average the chemical signature)
    if leaf_pixels.shape[0] == 0:
        signature = np.zeros((NUM_BANDS,NUM_STATS),dtype=np.float32)
    else:
        mean_signature = np.mean(leaf_pixels, axis=0)
        std_signature = np.std(leaf_pixels, axis=0)
        median_signature = np.median(leaf_pixels, axis=0)
        p25_signature = np.percentile(leaf_pixels, 25, axis=0)
        p75_signature = np.percentile(leaf_pixels, 75, axis=0)

        signature = np.stack([
                    mean_signature,
                    std_signature,
                    median_signature,
                    p25_signature,
                    p75_signature
                ],axis=-1)



    # Store the 1D vector
    X_data[i] = signature

print(f"Successfully processed {X_data.shape[0]} TIFF samples.")

print("Compressing and saving data to disk...")

np.savez_compressed("hyperleaf_processed.npz", X=X_data, y=y_data)
print("Saved successfully!")



Processing hyperspectral cubes into 1D spectral vectors...
Successfully processed 1590 TIFF samples.
Compressing and saving data to disk...
Saved successfully!


## MODEL ##


In [4]:
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU Detected and Ready: {gpus}\n")
    except RuntimeError as e:
        print(f"GPU configuration error: {e}\n")
else:
    print("No GPU found. Falling back to CPU execution.\n")

# ==========================================
# Shared Data Loading, Splitting, Normalizing
# ==========================================
loaded_data = np.load("hyperleaf_processed.npz")
X_data = loaded_data['X']
y_data = loaded_data['y']
NUM_STATS = X_data.shape[2]
CLASS_NAMES = ['Heerup', 'Kvium', 'Rembrandt', 'Sheriff']

print(f"Loaded data shape: {X_data.shape}")

X_train, X_temp, y_train, y_temp = train_test_split(
    X_data, y_data, test_size=0.2, random_state=42, stratify=y_data
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

X_min = X_train.min()
X_max = X_train.max()

X_train = (X_train - X_min) / (X_max - X_min + 1e-8)
X_val = (X_val - X_min) / (X_max - X_min + 1e-8)
X_test = (X_test - X_min) / (X_max - X_min + 1e-8)

print(f"Training shapes:   {X_train.shape} (80%)")
print(f"Validation shapes: {X_val.shape} (10%)")
print(f"Local Test shapes: {X_test.shape} (10%)")

def compile_and_train(model, model_name, epochs=200, batch_size=40):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    model.summary()

    checkpoint_path = f"best_{model_name}.keras"
    checkpoint = tf.keras.callbacks.ModelCheckpoint(
        checkpoint_path,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    )
    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=12,
        min_lr=1e-6,
        verbose=1
    )

    print(f"\nStarting training: {model_name}")
    with tf.device("/GPU:0"):
        history = model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=[checkpoint, reduce_lr]
        )

    best_val_accuracy = max(history.history["val_accuracy"])
    best_val_epoch = int(np.argmax(history.history["val_accuracy"]) + 1)
    print(f"Best validation accuracy: {best_val_accuracy * 100:.2f}% at epoch {best_val_epoch}")

    custom_objects = {}
    if "MambaBlock" in globals():
        custom_objects["MambaBlock"] = MambaBlock

    best_model = tf.keras.models.load_model(
        checkpoint_path,
        custom_objects=custom_objects
    )
    test_loss, test_accuracy = best_model.evaluate(X_test, y_test, verbose=0)
    print(f"\n{model_name} local test accuracy: {test_accuracy * 100:.2f}%")

    predictions_probs = best_model.predict(X_test, verbose=0)
    predicted_indices = np.argmax(predictions_probs, axis=1)
    print(classification_report(y_test, predicted_indices, target_names=CLASS_NAMES))

    return best_model, history


GPU Detected and Ready: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Loaded data shape: (1590, 204, 5)
Training shapes:   (1272, 204, 5) (80%)
Validation shapes: (159, 204, 5) (10%)
Local Test shapes: (159, 204, 5) (10%)


## CNN + MAMBA ##

In [68]:
class MambaBlock(layers.Layer):
    def __init__(self, d_model, d_state=64, d_conv=8, expand=8, **kwargs):
        super(MambaBlock, self).__init__(**kwargs)
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.expand = expand
        self.d_inner = int(self.expand * self.d_model)

    def build(self, input_shape):
        self.in_proj = layers.Dense(self.d_inner * 2, use_bias=False)
        self.conv1d = layers.Conv1D(
            filters=self.d_inner,
            kernel_size=self.d_conv,
            padding='same',
            groups=self.d_inner,
            activation='swish'
        )
        self.x_proj = layers.Dense(self.d_state * 2 + self.d_inner, use_bias=False)
        self.dt_proj = layers.Dense(self.d_inner, activation='softplus')
        self.out_proj = layers.Dense(self.d_model, use_bias=False)
        super(MambaBlock, self).build(input_shape)

    def call(self, x):
        # 1. Gated branch splitting
        projected = self.in_proj(x)
        x_branch, res_branch = tf.split(projected, num_or_size_splits=2, axis=-1)
        x_branch = self.conv1d(x_branch)

        # 2. Extract State Space parameters
        ssm_params = self.x_proj(x_branch)
        B, C, delta = tf.split(ssm_params, [self.d_state, self.d_state, self.d_inner], axis=-1)
        delta = self.dt_proj(delta)

        # 🌟 THE VECTORIZED FIX: No more sequential item loops 🌟
        # We compute the discretization factors globally using cumulative products
        # This replaces the iterative loop with parallel tensor math that fits perfectly on the GPU
        delta_x = delta * x_branch
        state_influence = tf.reduce_mean(B, axis=-1, keepdims=True)
        decay_factors = tf.exp(-delta * tf.abs(state_influence))

        # Cumulative scan parallelizes the recurrence sequence across CUDA cores instantly
        cum_decay = tf.math.cumprod(decay_factors, axis=1)
        outputs = tf.math.cumsum(delta_x * cum_decay, axis=1) / (cum_decay + 1e-8)

        # Final output projection
        outputs = outputs * tf.reduce_mean(C, axis=-1, keepdims=True)
        gated_output = outputs * tf.keras.activations.swish(res_branch)
        return self.out_proj(gated_output)

    def get_config(self):
        config = super(MambaBlock, self).get_config()
        config.update({
            "d_model": self.d_model,
            "d_state": self.d_state,
            "d_conv": self.d_conv,
            "expand": self.expand
        })
        return config
inputs = layers.Input(shape=(NUM_BANDS, NUM_STATS))

# Block 1: Initial Local Convolution Feature Aggregator
x = layers.Conv1D(128, kernel_size=5, activation='relu', padding='same')(inputs)
x = layers.MaxPooling1D(2)(x)
x = layers.LayerNormalization()(x)

# Block 2: The Mamba State-Space Model Layer
# d_model matches the 64 incoming feature channels from the Conv1D block
mamba_output = MambaBlock(d_model=128, d_state=16, expand=2)(x)

x = layers.Add()([x, mamba_output])
x = layers.LayerNormalization()(x)

x = layers.Conv1D(256, kernel_size=5, activation='relu', padding='same')(x)
x = layers.MaxPooling1D(2)(x)

x = layers.Flatten()(x)
x = layers.Dense(128, activation='relu')(x)

x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

cnn_mamba_model = models.Model(inputs=inputs, outputs=outputs)
cnn_mamba_model, cnn_mamba_history = compile_and_train(
    cnn_mamba_model,
    "hyper2_cnn_mamba",
    epochs=200
)
model = cnn_mamba_model
history = cnn_mamba_history

#91.82

Model: "model_44"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_45 (InputLayer)          [(None, 204, 5)]     0           []                               
                                                                                                  
 conv1d_88 (Conv1D)             (None, 204, 128)     3328        ['input_45[0][0]']               
                                                                                                  
 max_pooling1d_88 (MaxPooling1D  (None, 102, 128)    0           ['conv1d_88[0][0]']              
 )                                                                                                
                                                                                                  
 layer_normalization_88 (LayerN  (None, 102, 128)    256         ['max_pooling1d_88[0][0]']

## TEST FROM TRAIN ##

In [43]:
CLASS_NAMES = ['Heerup', 'Kvium', 'Rembrandt', 'Sheriff']

print("\n" + "="*60)
print("HOLDOUT TEST SET EVALUATION")
print("="*60)

predictions_probs = model.predict(X_test, verbose=0)
predicted_indices = np.argmax(predictions_probs, axis=1)

correct_count = 0
total_samples = len(y_test)

for i in range(total_samples):
    pred_idx = predicted_indices[i]
    true_idx = y_test[i]

    pred_name = CLASS_NAMES[pred_idx]
    true_name = CLASS_NAMES[true_idx]
    confidence = predictions_probs[i][pred_idx] * 100

    if pred_idx == true_idx:
        #status = "CORRECT"
        correct_count += 1
    """else:
        status = "WRONG"

    print(f"Sample #{i+1:03d} | Predicted: {pred_name: <10} ({confidence:.1f}%) | Actual: {true_name: <10} | {status}")"""

final_accuracy = (correct_count / total_samples) * 100
print(f"HOLDOUT ACCURACY: {correct_count}/{total_samples} ({final_accuracy:.2f}%)")

print("\nDETAILED CLASSIFICATION REPORT:")
print(classification_report(y_test, predicted_indices, target_names=CLASS_NAMES))


HOLDOUT TEST SET EVALUATION
HOLDOUT ACCURACY: 144/159 (90.57%)

DETAILED CLASSIFICATION REPORT:
              precision    recall  f1-score   support

      Heerup       0.90      0.90      0.90        41
       Kvium       0.91      0.83      0.87        35
   Rembrandt       0.84      0.95      0.89        43
     Sheriff       1.00      0.93      0.96        40

    accuracy                           0.91       159
   macro avg       0.91      0.90      0.91       159
weighted avg       0.91      0.91      0.91       159

